# 00 表格数据下载

**目的**：从 521K 条 Kaggle 元数据中筛选表格数据集候选，通过 Kaggle API 搜索可下载引用，批量下载并选择主表。

实现 CONTENT_VIEW_EXTENSION.md §2（数据采集与定位）。

**输入**：
- `data/metadata_merged.csv` — 合并后的元数据（约 521K 行）
- `data/raw_data/DatasetVersions.csv` — 数据集版本记录
- `tmp/index_map.parquet` — 全局 doc_idx 映射表

**输出**：
- `tmp/content/candidates.parquet` — 筛选后的候选数据集池
- `tmp/content/slug_to_ref.csv` — Kaggle API slug→ref 映射
- `tmp/content/d_content.parquet` — 最终 D_content 子集（B_ds 行）
- `tmp/content/main_tables.parquet` — 已选主表文件注册表
- `data/tabular_raw/{Id}/` — 已下载的数据集文件
- `tmp/content/api_cache/{slug}.json` — API 搜索结果缓存

In [1]:
# ---- 配置与导入 ----
import os, sys, json, re, time, subprocess, shutil, csv
from pathlib import Path
from io import StringIO

import numpy as np
import pandas as pd

# DatasetVersions.csv 包含超长 Description 字段，需提升 Python CSV 解析器的字段限制
csv.field_size_limit(sys.maxsize)

# 路径配置（notebook 位于 notebooks/04_content/，项目根目录在两级之上）
ROOT        = Path(".").resolve().parent.parent
DATA_DIR    = ROOT / "data"
RAW_DIR     = DATA_DIR / "raw_data"
TMP_DIR     = ROOT / "tmp"
CONTENT_DIR = TMP_DIR / "content"
CACHE_DIR   = CONTENT_DIR / "api_cache"
TAB_RAW_DIR = DATA_DIR / "tabular_raw"

for d in [CONTENT_DIR, CACHE_DIR, TAB_RAW_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# 确保 src 可导入
sys.path.insert(0, str(ROOT))
from src.content.sampling import select_main_table

# 预算参数
B_DS          = 1000                    # 主预算：要获取的数据集数量
MIN_VIEWS     = 1000                    # 最低 TotalViews 阈值
SIZE_MIN      = 100 * 1024              # 最小文件大小 100 KB
SIZE_MAX      = 100 * 1024 * 1024       # 最大文件大小 100 MB
SEARCH_MARGIN = 1.2                     # 搜索余量系数

# 表格相关标签集合
TABULAR_TAGS = {
    "tabular", "classification", "regression",
    "exploratory data analysis", "data analytics",
}

print(f"ROOT:        {ROOT}")
print(f"CONTENT_DIR: {CONTENT_DIR}")
print(f"TAB_RAW_DIR: {TAB_RAW_DIR}")
print(f"B_DS={B_DS}, MIN_VIEWS={MIN_VIEWS}, SIZE=[{SIZE_MIN/1024:.0f}KB, {SIZE_MAX/1024/1024:.0f}MB]")
print(f"SEARCH_MARGIN={SEARCH_MARGIN}")

ROOT:        /workspace/recsys-new
CONTENT_DIR: /workspace/recsys-new/tmp/content
TAB_RAW_DIR: /workspace/recsys-new/data/tabular_raw
B_DS=1000, MIN_VIEWS=1000, SIZE=[100KB, 100MB]
SEARCH_MARGIN=1.2


In [2]:
# ---- 加载元数据 ----
meta = pd.read_csv(DATA_DIR / "metadata_merged.csv", engine="python")
print(f"metadata_merged: {meta.shape}")

# 加载 DatasetVersions 以获取 TotalCompressedBytes
dv_cols = ["Id", "DatasetId", "VersionNumber", "TotalCompressedBytes"]
dv = pd.read_csv(RAW_DIR / "DatasetVersions.csv", usecols=dv_cols, engine="python")
print(f"DatasetVersions: {dv.shape}")

# 删除 DatasetId 或 VersionNumber 为 NaN 的行（数据质量问题）
dv = dv.dropna(subset=["DatasetId", "VersionNumber"])

# 保留每个数据集的最新版本
idx_latest = dv.groupby("DatasetId")["VersionNumber"].idxmax()
dv_latest = dv.loc[idx_latest, ["DatasetId", "TotalCompressedBytes"]].copy()
dv_latest.rename(columns={"DatasetId": "Id"}, inplace=True)
print(f"Latest versions: {dv_latest.shape}")

# 关联以获取文件大小信息
meta = meta.merge(dv_latest, on="Id", how="left")
print(f"After join: {meta.shape}, TotalCompressedBytes notna: {meta['TotalCompressedBytes'].notna().sum()}")
del dv  # 释放内存

metadata_merged: (521735, 31)


DatasetVersions: (1331684, 4)
Latest versions: (521233, 2)


After join: (521735, 32), TotalCompressedBytes notna: 521214


In [3]:
# ---- 筛选候选集 ----
def has_tabular_tag(tags_str):
    """检查逗号分隔的 Tags 字符串中是否包含表格相关标签。"""
    if pd.isna(tags_str) or not isinstance(tags_str, str):
        return False
    tags = {t.strip().lower() for t in tags_str.split(",")}
    return bool(tags & TABULAR_TAGS)


def filter_candidates(df, min_views=MIN_VIEWS, size_min=SIZE_MIN, size_max=SIZE_MAX):
    """对数据集施加标签、大小和浏览量筛选。"""
    mask_tag = df["Tags"].apply(has_tabular_tag)
    mask_size = (
        df["TotalCompressedBytes"].notna()
        & (df["TotalCompressedBytes"] >= size_min)
        & (df["TotalCompressedBytes"] <= size_max)
    )
    mask_views = df["TotalViews"] >= min_views

    print(f"Tag filter:   {mask_tag.sum():>7,} datasets")
    print(f"Size filter:  {mask_size.sum():>7,} datasets")
    print(f"Views filter: {mask_views.sum():>7,} datasets")

    combined = mask_tag & mask_size & mask_views
    print(f"Combined:     {combined.sum():>7,} datasets")
    return df[combined].copy()


candidates = filter_candidates(meta)

# 报告不同预算下的候选池大小
print("\nPool sizes for different B_ds budgets:")
for b in [500, 1000, 2000]:
    pool = candidates.nlargest(b, "TotalDownloads")
    print(f"  B_ds={b:>5d}: {len(pool):>5d} datasets, "
          f"min views={pool['TotalViews'].min():,.0f}, "
          f"min downloads={pool['TotalDownloads'].min():,.0f}")

# 保存候选集
candidates.to_parquet(CONTENT_DIR / "candidates.parquet", engine="fastparquet")
print(f"\nSaved candidates.parquet: {len(candidates)} rows")

Tag filter:    25,833 datasets
Size filter:  208,935 datasets
Views filter: 120,152 datasets
Combined:       7,393 datasets

Pool sizes for different B_ds budgets:
  B_ds=  500:   500 datasets, min views=2,816, min downloads=6,132
  B_ds= 1000:  1000 datasets, min views=1,777, min downloads=3,002
  B_ds= 2000:  2000 datasets, min views=1,777, min downloads=1,419

Saved candidates.parquet: 7393 rows


In [4]:
# ---- Kaggle API 可用性检测 ----
def check_kaggle_api():
    """测试 kaggle CLI 是否可用且已认证。"""
    try:
        result = subprocess.run(
            ["kaggle", "--version"],
            capture_output=True, text=True, timeout=10
        )
        if result.returncode != 0:
            print(f"Kaggle CLI error: {result.stderr.strip()}")
            return False
        print(f"Kaggle CLI available: {result.stdout.strip()}")
    except FileNotFoundError:
        print("Kaggle CLI not found. Install via: pip install kaggle")
        return False
    except Exception as e:
        print(f"Kaggle CLI check failed: {e}")
        return False

    # 验证凭证：执行一次轻量 API 调用
    try:
        result = subprocess.run(
            ["kaggle", "datasets", "list", "-s", "titanic", "--csv", "--max-size", "1048576"],
            capture_output=True, text=True, timeout=30
        )
        if result.returncode != 0:
            print(f"Kaggle API auth failed: {result.stderr.strip()[:200]}")
            print("Place kaggle.json in ~/.kaggle/ (or ~/.config/kaggle/) and retry.")
            return False
        print("Kaggle API authenticated successfully.")
        return True
    except Exception as e:
        print(f"Kaggle API auth check failed: {e}")
        return False


KAGGLE_AVAILABLE = check_kaggle_api()

Kaggle CLI available: Kaggle API 1.7.4.5


Kaggle API authenticated successfully.


In [5]:
# ---- API 搜索函数 ----
def search_kaggle_slug(slug, max_size_mb=100):
    """
    通过 Kaggle API 搜索与 slug 匹配的数据集。
    使用 CACHE_DIR 中按 slug 命名的缓存文件，避免重复 API 调用。
    """
    cache_file = CACHE_DIR / f"{slug}.json"
    if cache_file.exists():
        with open(cache_file) as f:
            return json.load(f)

    try:
        max_size_bytes = max_size_mb * 1024 * 1024
        cmd = [
            "kaggle", "datasets", "list",
            "-s", slug,
            "--csv",
            "--max-size", str(max_size_bytes),
        ]
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
        time.sleep(0.5)  # 速率限制

        if result.returncode != 0 or not result.stdout.strip():
            results = []
        else:
            csv_text = result.stdout.strip()
            if csv_text.startswith("ref") or csv_text.startswith('"ref'):
                df = pd.read_csv(StringIO(csv_text))
                results = df.to_dict("records")
            else:
                results = []
    except Exception as e:
        print(f"  API error for '{slug}': {e}")
        results = []

    with open(cache_file, "w") as f:
        json.dump(results, f)

    return results

In [6]:
# ---- Slug→Ref 匹配 ----
def match_slug_to_ref(slug, title, api_results):
    """
    将本地 slug 匹配到 Kaggle API 的 ref。
    优先级：ref 的精确后缀匹配，然后是标题模糊回退。
    返回 (ref, confidence)，confidence 为 'exact'|'fuzzy'|'none'。
    """
    if not api_results:
        return None, "none"

    slug_lower = slug.lower().strip()

    # 1. 精确后缀匹配
    for item in api_results:
        ref = str(item.get("ref", "")).lower().strip()
        if ref.endswith(f"/{slug_lower}"):
            return item["ref"], "exact"

    # 2. 标题模糊回退
    if title and isinstance(title, str):
        title_lower = title.lower().strip()
        for item in api_results:
            api_title = str(item.get("title", "")).lower().strip()
            if title_lower == api_title or title_lower in api_title or api_title in title_lower:
                return item["ref"], "fuzzy"

    return None, "none"

In [7]:
# ---- 批量搜索（可续传） ----
slug_ref_path = CONTENT_DIR / "slug_to_ref.csv"

if not KAGGLE_AVAILABLE:
    print("Kaggle API not available. To enable:")
    print("  1. pip install kaggle")
    print("  2. Place kaggle.json in ~/.kaggle/")
    print("  3. Re-run this cell")
    print("")
    print("Checking for existing slug_to_ref.csv to resume from...")

# 加载已有结果
if slug_ref_path.exists():
    slug_to_ref = pd.read_csv(slug_ref_path)
    print(f"Loaded existing slug_to_ref.csv: {len(slug_to_ref)} rows")
else:
    slug_to_ref = pd.DataFrame(columns=["Id", "Slug", "Title", "ref", "confidence"])

if KAGGLE_AVAILABLE:
    n_search = int(B_DS * SEARCH_MARGIN)
    search_pool = candidates.nlargest(n_search, "TotalDownloads")

    already_searched = set(slug_to_ref["Slug"].values) if len(slug_to_ref) > 0 else set()
    to_search = search_pool[~search_pool["Slug"].isin(already_searched)]
    print(f"Search pool: {len(search_pool)}, already searched: {len(already_searched)}, remaining: {len(to_search)}")

    new_rows = []
    for i, (_, row) in enumerate(to_search.iterrows()):
        slug = row["Slug"]
        title = row["Title"]
        ds_id = row["Id"]

        results = search_kaggle_slug(slug)
        ref, confidence = match_slug_to_ref(slug, title, results)

        new_rows.append({
            "Id": ds_id,
            "Slug": slug,
            "Title": title,
            "ref": ref if ref else "",
            "confidence": confidence,
        })

        # 每 50 条增量保存（改进：NB1 原版仅在结束时保存）
        if (i + 1) % 50 == 0:
            incremental_df = pd.DataFrame(new_rows)
            slug_to_ref = pd.concat([slug_to_ref, incremental_df], ignore_index=True)
            slug_to_ref.to_csv(slug_ref_path, index=False)
            new_rows = []  # 重置缓冲区
            matched_so_far = (slug_to_ref["confidence"] != "none").sum()
            print(f"  Searched {i+1}/{len(to_search)} "
                  f"(total matched: {matched_so_far}), saved checkpoint")

    # 保存剩余记录
    if new_rows:
        new_df = pd.DataFrame(new_rows)
        slug_to_ref = pd.concat([slug_to_ref, new_df], ignore_index=True)
        slug_to_ref.to_csv(slug_ref_path, index=False)

    print(f"\nSaved slug_to_ref.csv: {len(slug_to_ref)} total rows")

# 确保 slug_to_ref.csv 产物始终存在（降级路径下保存空文件）
if not slug_ref_path.exists():
    slug_to_ref.to_csv(slug_ref_path, index=False)
    print("Saved empty slug_to_ref.csv as placeholder")

# 汇总统计
if len(slug_to_ref) > 0:
    print(f"\nConfidence breakdown:")
    print(slug_to_ref["confidence"].value_counts().to_string())
else:
    print("No slug_to_ref data available. Kaggle API search needed.")

Loaded existing slug_to_ref.csv: 1200 rows
Search pool: 1200, already searched: 1161, remaining: 0

Saved slug_to_ref.csv: 1200 total rows

Confidence breakdown:
confidence
exact    1171
none       21
fuzzy       8


In [8]:
# ---- 构建 d_content 子集 ----
index_map = pd.read_parquet(TMP_DIR / "index_map.parquet", engine="fastparquet")

if len(slug_to_ref) > 0 and (slug_to_ref["confidence"] != "none").any():
    matched = slug_to_ref[slug_to_ref["confidence"] != "none"].copy()
    matched["Id"] = matched["Id"].astype(int)

    matched = matched.merge(
        candidates[["Id", "TotalDownloads", "TotalViews"]],
        on="Id", how="left"
    )

    d_content = matched.nlargest(B_DS, "TotalDownloads").copy()
    d_content = d_content.merge(index_map, on="Id", how="left")
    d_content = d_content.dropna(subset=["doc_idx"])
    d_content["doc_idx"] = d_content["doc_idx"].astype(int)

    d_content = d_content[["doc_idx", "Id", "Slug", "ref", "TotalDownloads", "TotalViews"]].copy()
    d_content = d_content.reset_index(drop=True)
else:
    print("No Kaggle API matches available. Building d_content from candidates directly.")
    top_cands = candidates.nlargest(B_DS, "TotalDownloads").copy()
    top_cands = top_cands.merge(index_map, on="Id", how="left")
    top_cands = top_cands.dropna(subset=["doc_idx"])
    top_cands["doc_idx"] = top_cands["doc_idx"].astype(int)

    d_content = top_cands[["doc_idx", "Id", "Slug", "TotalDownloads", "TotalViews"]].copy()
    d_content["ref"] = ""
    d_content = d_content[["doc_idx", "Id", "Slug", "ref", "TotalDownloads", "TotalViews"]]
    d_content = d_content.reset_index(drop=True)

d_content.to_parquet(CONTENT_DIR / "d_content.parquet", engine="fastparquet")
print(f"d_content.parquet: {len(d_content)} rows")
print(f"  doc_idx range: [{d_content['doc_idx'].min()}, {d_content['doc_idx'].max()}]")
print(f"  TotalDownloads range: [{d_content['TotalDownloads'].min():,.0f}, {d_content['TotalDownloads'].max():,.0f}]")
d_content.head()

d_content.parquet: 1000 rows
  doc_idx range: [551, 519823]
  TotalDownloads range: [2,967, 626,556]


,doc_idx,Id,Slug,ref,TotalDownloads,TotalViews
0,24378,434238,netflix-shows,shivamb/netflix-shows,626556,3538116
1,4257,11167,mobile-price-classification,iabhishekofficial/mobile-price-classification,230776,1112565
2,188218,2818963,amazon-sales-dataset,karkavelrajaj/amazon-sales-dataset,187755,742015
3,3096,6012,weather-dataset-rattle-package,jsphyg/weather-dataset-rattle-package,159026,824725
4,129807,1940216,superstore-dataset-final,vivek468/superstore-dataset-final,154536,607004


In [9]:
# ---- 下载函数 ----
def download_dataset(ref, dataset_id, output_dir):
    """
    使用 Kaggle CLI 下载数据集。
    封装命令：kaggle datasets download -d {ref} -p {dir} --unzip
    成功返回 True，失败返回 False。
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    if not ref or ref == "":
        return False

    try:
        cmd = [
            "kaggle", "datasets", "download",
            "-d", ref,
            "-p", str(output_dir),
            "--unzip",
        ]
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
        if result.returncode != 0:
            print(f"  Download failed for {ref}: {result.stderr[:200]}")
            return False
        return True
    except subprocess.TimeoutExpired:
        print(f"  Download timeout for {ref}")
        return False
    except Exception as e:
        print(f"  Download error for {ref}: {e}")
        return False

In [10]:
# ---- 批量下载（可续传 + 重试） ----
MAX_RETRIES = 2

if KAGGLE_AVAILABLE and (d_content["ref"] != "").any():
    to_download = d_content[d_content["ref"] != ""].copy()
    n_downloaded = 0
    n_skipped = 0
    n_failed = 0
    failed_list = []

    for i, (_, row) in enumerate(to_download.iterrows()):
        ds_id = int(row["Id"])
        ref = row["ref"]
        ds_dir = TAB_RAW_DIR / str(ds_id)

        # 如果目录已存在且非空则跳过
        if ds_dir.exists() and any(ds_dir.iterdir()):
            n_skipped += 1
            continue

        # 重试逻辑（指数退避）
        success = False
        for attempt in range(MAX_RETRIES + 1):
            if attempt > 0:
                wait = 2 ** attempt
                print(f"  Retry {attempt}/{MAX_RETRIES} for {ref} after {wait}s")
                time.sleep(wait)
            success = download_dataset(ref, ds_id, ds_dir)
            if success:
                break

        if success:
            n_downloaded += 1
        else:
            n_failed += 1
            failed_list.append({"Id": ds_id, "ref": ref})

        if (i + 1) % 50 == 0:
            print(f"  Progress: {i+1}/{len(to_download)} "
                  f"(downloaded={n_downloaded}, skipped={n_skipped}, failed={n_failed})")

    print(f"\nDownload complete: {n_downloaded} new, {n_skipped} skipped, {n_failed} failed")
    if failed_list:
        print(f"Failed datasets ({len(failed_list)}):")
        for f in failed_list[:20]:
            print(f"  Id={f['Id']}, ref={f['ref']}")
else:
    print("Skipping downloads: Kaggle API not available or no refs matched.")
    print("Place dataset files manually in data/tabular_raw/{DatasetId}/ to continue.")


Download complete: 0 new, 1000 skipped, 0 failed


In [11]:
# ---- 主表选择 ----
table_rows = []
for idx, (_, row) in enumerate(d_content.iterrows()):
    ds_id = int(row["Id"])
    doc_idx = int(row["doc_idx"])
    ds_dir = TAB_RAW_DIR / str(ds_id)

    path, fsize, ext = select_main_table(ds_dir)
    table_rows.append({
        "DatasetId": ds_id,
        "doc_idx": doc_idx,
        "main_table_path": path if path else "",
        "file_size": fsize,
        "extension": ext if ext else "",
    })

main_tables = pd.DataFrame(table_rows)
main_tables.to_parquet(CONTENT_DIR / "main_tables.parquet", engine="fastparquet")

# 覆盖率报告
n_found = (main_tables["main_table_path"] != "").sum()
n_total = len(main_tables)
pct = n_found / n_total * 100 if n_total > 0 else 0.0
print(f"main_tables.parquet: {n_total} datasets")
print(f"  Tables found: {n_found} ({pct:.1f}%)")
print(f"  Missing:      {n_total - n_found}")

if n_found > 0:
    found = main_tables[main_tables["main_table_path"] != ""]
    print(f"  Extension breakdown:")
    print(found["extension"].value_counts().to_string(header=False))
    print(f"  File size: median={found['file_size'].median()/1024:.1f} KB, "
          f"max={found['file_size'].max()/1024/1024:.1f} MB")

main_tables.parquet: 1000 datasets
  Tables found: 933 (93.3%)
  Missing:      67
  Extension breakdown:
.csv        876
.xlsx        45
.xls          6
.parquet      5
.tsv          1
  File size: median=3227.7 KB, max=1035.7 MB


In [12]:
# ---- 非表格数据集回填 ----
# 识别 main_table_path == "" 的数据集，从 d_content 中移除，
# 然后从 Tier-1（已匹配但未选入的缓冲区）和 Tier-2（未搜索候选）中依次递补。

non_tabular_mask = main_tables["main_table_path"] == ""
n_non_tabular = non_tabular_mask.sum()
print(f"非表格数据集: {n_non_tabular}")

if n_non_tabular > 0:
    non_tab_ids = set(main_tables.loc[non_tabular_mask, "DatasetId"].astype(int).values)
    print(f"  待移除 IDs (前 10): {sorted(non_tab_ids)[:10]}...")

    # --- 移除非表格数据集 ---
    d_content = d_content[~d_content["Id"].astype(int).isin(non_tab_ids)].reset_index(drop=True)
    main_tables = main_tables[~main_tables["DatasetId"].astype(int).isin(non_tab_ids)].reset_index(drop=True)
    print(f"  移除后: d_content={len(d_content)}, main_tables={len(main_tables)}")

    need = B_DS - len(d_content)
    print(f"  需要回填: {need}")

    # --- Tier-1: 已匹配但未选入的缓冲区 ---
    dc_ids = set(d_content["Id"].astype(int).values)
    tier1_pool = slug_to_ref[
        (slug_to_ref["confidence"] != "none")
        & (~slug_to_ref["Id"].astype(int).isin(dc_ids))
        & (~slug_to_ref["Id"].astype(int).isin(non_tab_ids))
    ].copy()
    # 按 TotalDownloads 排序（需要合并候选集信息）
    tier1_pool = tier1_pool.merge(
        candidates[["Id", "TotalDownloads", "TotalViews"]],
        on="Id", how="left"
    )
    tier1_pool = tier1_pool.sort_values("TotalDownloads", ascending=False).reset_index(drop=True)
    print(f"  Tier-1 缓冲区大小: {len(tier1_pool)}")

    backfill_rows_dc = []
    backfill_rows_mt = []
    n_tier1_tried = 0
    n_tier1_ok = 0
    n_tier1_skip = 0

    for _, brow in tier1_pool.iterrows():
        if need <= 0:
            break
        n_tier1_tried += 1
        ds_id = int(brow["Id"])
        ref = brow["ref"]
        slug = brow["Slug"]
        ds_dir = TAB_RAW_DIR / str(ds_id)

        # 下载（跳过已有非空目录）
        if ds_dir.exists() and any(ds_dir.iterdir()):
            pass  # 已下载
        else:
            ok = download_dataset(ref, ds_id, ds_dir)
            if not ok:
                print(f"    Tier-1 下载失败: {ref}")
                n_tier1_skip += 1
                continue

        # 验证是否有表格文件
        path, fsize, ext = select_main_table(ds_dir)
        if path is None:
            # 非表格，删除目录
            if ds_dir.exists():
                shutil.rmtree(ds_dir)
            n_tier1_skip += 1
            continue

        # 获取 doc_idx
        idx_row = index_map[index_map["Id"] == ds_id]
        if len(idx_row) == 0:
            n_tier1_skip += 1
            continue
        doc_idx = int(idx_row["doc_idx"].iloc[0])

        # 成功！加入回填列表
        backfill_rows_dc.append({
            "doc_idx": doc_idx,
            "Id": ds_id,
            "Slug": slug,
            "ref": ref,
            "TotalDownloads": brow["TotalDownloads"],
            "TotalViews": brow["TotalViews"],
        })
        backfill_rows_mt.append({
            "DatasetId": ds_id,
            "doc_idx": doc_idx,
            "main_table_path": path,
            "file_size": fsize,
            "extension": ext,
        })
        need -= 1
        n_tier1_ok += 1

        if n_tier1_ok % 20 == 0:
            print(f"    Tier-1 进度: 已回填 {n_tier1_ok}, 跳过 {n_tier1_skip}, 剩余 {need}")

    print(f"  Tier-1 完成: 尝试 {n_tier1_tried}, 成功 {n_tier1_ok}, 跳过 {n_tier1_skip}")

    # --- Tier-2: 未搜索候选（仅在 Tier-1 不足时） ---
    n_tier2_ok = 0
    n_tier2_skip = 0
    tier2_new_slug_rows = []

    if need > 0 and KAGGLE_AVAILABLE:
        print(f"\n  Tier-1 不足，启动 Tier-2 搜索 (还需 {need})...")
        already_ids = dc_ids | set(r["Id"] for r in backfill_rows_dc) | non_tab_ids
        already_slugs = set(slug_to_ref["Slug"].values)
        tier2_pool = candidates[
            (~candidates["Id"].isin(already_ids))
            & (~candidates["Slug"].isin(already_slugs))
        ].nlargest(need * 5, "TotalDownloads")  # 搜索 5 倍余量
        print(f"  Tier-2 候选池: {len(tier2_pool)}")

        for _, crow in tier2_pool.iterrows():
            if need <= 0:
                break
            slug = crow["Slug"]
            title = crow["Title"]
            ds_id = int(crow["Id"])

            # API 搜索
            results = search_kaggle_slug(slug)
            ref, confidence = match_slug_to_ref(slug, title, results)
            tier2_new_slug_rows.append({
                "Id": ds_id, "Slug": slug, "Title": title,
                "ref": ref if ref else "", "confidence": confidence,
            })

            if confidence == "none" or not ref:
                n_tier2_skip += 1
                continue

            ds_dir = TAB_RAW_DIR / str(ds_id)
            if ds_dir.exists() and any(ds_dir.iterdir()):
                pass
            else:
                ok = download_dataset(ref, ds_id, ds_dir)
                if not ok:
                    n_tier2_skip += 1
                    continue

            path, fsize, ext = select_main_table(ds_dir)
            if path is None:
                if ds_dir.exists():
                    shutil.rmtree(ds_dir)
                n_tier2_skip += 1
                continue

            idx_row = index_map[index_map["Id"] == ds_id]
            if len(idx_row) == 0:
                n_tier2_skip += 1
                continue
            doc_idx = int(idx_row["doc_idx"].iloc[0])

            backfill_rows_dc.append({
                "doc_idx": doc_idx, "Id": ds_id, "Slug": slug,
                "ref": ref, "TotalDownloads": crow["TotalDownloads"],
                "TotalViews": crow["TotalViews"],
            })
            backfill_rows_mt.append({
                "DatasetId": ds_id, "doc_idx": doc_idx,
                "main_table_path": path, "file_size": fsize, "extension": ext,
            })
            need -= 1
            n_tier2_ok += 1

        print(f"  Tier-2 完成: 成功 {n_tier2_ok}, 跳过 {n_tier2_skip}")

    # --- 合并回填结果 ---
    if backfill_rows_dc:
        new_dc = pd.DataFrame(backfill_rows_dc)
        new_mt = pd.DataFrame(backfill_rows_mt)
        d_content = pd.concat([d_content, new_dc], ignore_index=True)
        main_tables = pd.concat([main_tables, new_mt], ignore_index=True)

    # --- 清理非表格数据集目录 ---
    n_cleaned = 0
    for nid in non_tab_ids:
        ndir = TAB_RAW_DIR / str(nid)
        if ndir.exists():
            shutil.rmtree(ndir)
            n_cleaned += 1
    print(f"\n  已清理非表格目录: {n_cleaned}")

    # --- 更新 slug_to_ref（Tier-2 新增） ---
    if tier2_new_slug_rows:
        new_sr = pd.DataFrame(tier2_new_slug_rows)
        slug_to_ref = pd.concat([slug_to_ref, new_sr], ignore_index=True)
        slug_to_ref.to_csv(slug_ref_path, index=False)
        print(f"  更新 slug_to_ref.csv: {len(slug_to_ref)} 行")

    # --- 保存更新后的产物 ---
    d_content.to_parquet(CONTENT_DIR / "d_content.parquet", engine="fastparquet")
    main_tables.to_parquet(CONTENT_DIR / "main_tables.parquet", engine="fastparquet")
    print(f"\n回填完成:")
    print(f"  d_content:    {len(d_content)} 行")
    print(f"  main_tables:  {len(main_tables)} 行")
    print(f"  有主表:       {(main_tables['main_table_path'] != '').sum()}")
    print(f"  回填成功:     Tier-1={n_tier1_ok}, Tier-2={n_tier2_ok}")
else:
    print("所有数据集均为表格型，无需回填。")

非表格数据集: 67
  待移除 IDs (前 10): [np.int64(6799), np.int64(20173), np.int64(22567), np.int64(30764), np.int64(32526), np.int64(36310), np.int64(42853), np.int64(50067), np.int64(143448), np.int64(164979)]...
  移除后: d_content=933, main_tables=933
  需要回填: 67
  Tier-1 缓冲区大小: 179


    Tier-1 进度: 已回填 20, 跳过 0, 剩余 47


    Tier-1 进度: 已回填 40, 跳过 1, 剩余 27


    Tier-1 进度: 已回填 60, 跳过 1, 剩余 7


  Tier-1 完成: 尝试 68, 成功 67, 跳过 1



  已清理非表格目录: 67

回填完成:
  d_content:    1000 行
  main_tables:  1000 行
  有主表:       1000
  回填成功:     Tier-1=67, Tier-2=0


In [13]:
# ---- 回填后完整性检查 ----
print("=" * 60)
print("回填后完整性检查")
print("=" * 60)

# 重新加载以确保磁盘数据与内存一致
_dc = pd.read_parquet(CONTENT_DIR / "d_content.parquet", engine="fastparquet")
_mt = pd.read_parquet(CONTENT_DIR / "main_tables.parquet", engine="fastparquet")
errors = []

# 1. 行数
print(f"\n[1] 行数检查:")
print(f"  d_content:   {len(_dc)} (目标 {B_DS})")
print(f"  main_tables: {len(_mt)} (目标 {B_DS})")
if len(_dc) != B_DS:
    errors.append(f"d_content 行数 {len(_dc)} != {B_DS}")
if len(_mt) != B_DS:
    errors.append(f"main_tables 行数 {len(_mt)} != {B_DS}")

# 2. ID 集合一致
print(f"\n[2] ID 一致性:")
dc_ids = set(_dc["Id"].astype(int).values)
mt_ids = set(_mt["DatasetId"].astype(int).values)
in_dc_not_mt = dc_ids - mt_ids
in_mt_not_dc = mt_ids - dc_ids
print(f"  d_content IDs: {len(dc_ids)}")
print(f"  main_tables IDs: {len(mt_ids)}")
print(f"  在 d_content 但不在 main_tables: {len(in_dc_not_mt)}")
print(f"  在 main_tables 但不在 d_content: {len(in_mt_not_dc)}")
if in_dc_not_mt:
    errors.append(f"d_content 有 {len(in_dc_not_mt)} 个 ID 不在 main_tables 中")
if in_mt_not_dc:
    errors.append(f"main_tables 有 {len(in_mt_not_dc)} 个 ID 不在 d_content 中")

# 3. 无重复 ID
print(f"\n[3] 重复 ID 检查:")
dc_dupes = _dc["Id"].duplicated().sum()
mt_dupes = _mt["DatasetId"].duplicated().sum()
print(f"  d_content 重复: {dc_dupes}")
print(f"  main_tables 重复: {mt_dupes}")
if dc_dupes > 0:
    errors.append(f"d_content 有 {dc_dupes} 个重复 ID")
if mt_dupes > 0:
    errors.append(f"main_tables 有 {mt_dupes} 个重复 ID")

# 4. 所有 main_table_path 非空
print(f"\n[4] 主表路径检查:")
n_empty = (_mt["main_table_path"] == "").sum()
n_nonempty = (_mt["main_table_path"] != "").sum()
print(f"  有主表: {n_nonempty}")
print(f"  无主表: {n_empty}")
if n_empty > 0:
    errors.append(f"仍有 {n_empty} 个数据集无主表")

# 5. 所有数据集目录存在且非空
print(f"\n[5] 目录存在性检查:")
n_missing_dir = 0
n_empty_dir = 0
for did in dc_ids:
    ddir = TAB_RAW_DIR / str(did)
    if not ddir.exists():
        n_missing_dir += 1
    elif not any(ddir.iterdir()):
        n_empty_dir += 1
print(f"  缺失目录: {n_missing_dir}")
print(f"  空目录: {n_empty_dir}")
if n_missing_dir > 0:
    errors.append(f"{n_missing_dir} 个数据集目录缺失")
if n_empty_dir > 0:
    errors.append(f"{n_empty_dir} 个数据集目录为空")

# 6. doc_idx 与 index_map 一致
print(f"\n[6] doc_idx 一致性:")
_imap = pd.read_parquet(TMP_DIR / "index_map.parquet", engine="fastparquet")
imap_lookup = dict(zip(_imap["Id"].astype(int), _imap["doc_idx"].astype(int)))
n_mismatch = 0
for _, row in _dc.iterrows():
    did = int(row["Id"])
    expected = imap_lookup.get(did)
    if expected is not None and int(row["doc_idx"]) != expected:
        n_mismatch += 1
print(f"  doc_idx 不匹配: {n_mismatch}")
if n_mismatch > 0:
    errors.append(f"{n_mismatch} 个 doc_idx 与 index_map 不一致")

# 7. 非表格目录已清理
print(f"\n[7] 非表格目录清理检查:")
existing_dirs = {int(d.name) for d in TAB_RAW_DIR.iterdir() if d.is_dir() and d.name.isdigit()}
stale_dirs = existing_dirs - dc_ids
print(f"  tabular_raw 目录数: {len(existing_dirs)}")
print(f"  不在 d_content 中的目录: {len(stale_dirs)}")

# 结果汇总
print(f"\n{'=' * 60}")
if errors:
    print(f"发现 {len(errors)} 个问题:")
    for e in errors:
        print(f"  ✗ {e}")
else:
    print("✓ 所有检查通过！d_content 全部为表格型数据集。")

回填后完整性检查

[1] 行数检查:
  d_content:   1000 (目标 1000)
  main_tables: 1000 (目标 1000)

[2] ID 一致性:
  d_content IDs: 1000
  main_tables IDs: 1000
  在 d_content 但不在 main_tables: 0
  在 main_tables 但不在 d_content: 0

[3] 重复 ID 检查:
  d_content 重复: 0
  main_tables 重复: 0

[4] 主表路径检查:
  有主表: 1000
  无主表: 0

[5] 目录存在性检查:
  缺失目录: 0
  空目录: 0

[6] doc_idx 一致性:


  doc_idx 不匹配: 0

[7] 非表格目录清理检查:
  tabular_raw 目录数: 1000
  不在 d_content 中的目录: 0

✓ 所有检查通过！d_content 全部为表格型数据集。


In [14]:
# ---- 端到端验证 ----
print("=" * 60)
print("端到端验证")
print("=" * 60)

# 1. 产物文件存在性与大小
artifacts = {
    "candidates.parquet": CONTENT_DIR / "candidates.parquet",
    "slug_to_ref.csv": CONTENT_DIR / "slug_to_ref.csv",
    "d_content.parquet": CONTENT_DIR / "d_content.parquet",
    "main_tables.parquet": CONTENT_DIR / "main_tables.parquet",
}
print("\n[1] 产物文件检查:")
for name, path in artifacts.items():
    if path.exists():
        size_kb = path.stat().st_size / 1024
        print(f"  \u2713 {name}: {size_kb:.1f} KB")
    else:
        print(f"  \u2717 {name}: MISSING")

# 2. d_content vs tabular_raw 目录交叉验证
print("\n[2] 下载覆盖验证:")
dc = pd.read_parquet(CONTENT_DIR / "d_content.parquet", engine="fastparquet")
dc_ids = set(dc["Id"].astype(int).values)
existing_dirs = set()
empty_dirs = set()
if TAB_RAW_DIR.exists():
    for d in TAB_RAW_DIR.iterdir():
        if d.is_dir():
            try:
                did = int(d.name)
            except ValueError:
                continue
            if any(d.iterdir()):
                existing_dirs.add(did)
            else:
                empty_dirs.add(did)

downloaded = dc_ids & existing_dirs
missing = dc_ids - existing_dirs - empty_dirs
print(f"  d_content IDs: {len(dc_ids)}")
print(f"  Downloaded (non-empty): {len(downloaded)}")
print(f"  Empty dirs: {len(dc_ids & empty_dirs)}")
print(f"  Missing dirs: {len(missing)}")

# 3. doc_idx vs index_map 一致性
print("\n[3] doc_idx 一致性检查:")
imap = pd.read_parquet(TMP_DIR / "index_map.parquet", engine="fastparquet")
imap_ids = set(imap["Id"].values)
dc_in_imap = dc_ids & set(imap["Id"].astype(int).values)
print(f"  d_content IDs in index_map: {len(dc_in_imap)}/{len(dc_ids)}")

# 4. 管线 yield 漏斗
print("\n[4] 管线 Yield 漏斗:")
mt = pd.read_parquet(CONTENT_DIR / "main_tables.parquet", engine="fastparquet")
n_candidates = len(pd.read_parquet(CONTENT_DIR / "candidates.parquet", engine="fastparquet"))
n_matched = (slug_to_ref["confidence"] != "none").sum() if len(slug_to_ref) > 0 else 0
n_dcontent = len(dc)
n_downloaded = len(downloaded)
n_has_table = (mt["main_table_path"] != "").sum()

print(f"  候选池:       {n_candidates:>6,}")
print(f"  API 匹配:     {n_matched:>6,}")
print(f"  d_content:    {n_dcontent:>6,}")
print(f"  已下载:       {n_downloaded:>6,}")
print(f"  有主表:       {n_has_table:>6,}")
print("\n验证完成.")

端到端验证

[1] 产物文件检查:
  ✓ candidates.parquet: 7096.7 KB
  ✓ slug_to_ref.csv: 131.3 KB
  ✓ d_content.parquet: 63.8 KB
  ✓ main_tables.parquet: 42.4 KB

[2] 下载覆盖验证:
  d_content IDs: 1000
  Downloaded (non-empty): 1000
  Empty dirs: 0
  Missing dirs: 0

[3] doc_idx 一致性检查:


  d_content IDs in index_map: 1000/1000

[4] 管线 Yield 漏斗:


  候选池:        7,393
  API 匹配:      1,179
  d_content:     1,000
  已下载:        1,000
  有主表:        1,000

验证完成.


## 总结

**产出文件**：
- `tmp/content/candidates.parquet` — 通过标签/大小/浏览量筛选的候选数据集
- `tmp/content/slug_to_ref.csv` — Kaggle slug → API ref 映射及匹配置信度
- `tmp/content/d_content.parquet` — 最终 D_content 子集（按下载量排名前 B_ds 个）
- `tmp/content/main_tables.parquet` — 已选主表文件注册表
- `data/tabular_raw/{Id}/` — 已下载的数据集目录
- `tmp/content/api_cache/{slug}.json` — API 搜索缓存

**下一步**：运行 `01_content_data_acquisition.ipynb` 验证数据质量，然后运行 `02_content_view_construction.ipynb` 进行表格剖析和内容嵌入构建。